# Income Classification with Logistic Regression – Practice Skeleton

**Short name (GitHub):** `Income_LogReg`  
**Lab source:** Codecademy *Income Classification using Logistic Regression* (UCI Adult / 1994 Census).  
**Data:** `data/adult.data` (no header; 32,561 rows).  
**Companion files:** `Income_LogReg_Solution.ipynb`, `Income_LogReg_Reusable_Template.ipynb`, `Income_LogReg_Cheatsheet.docx`, `Income_LogReg_Project_Memo.docx`, `Income_LogReg_Strategy_Guide.docx`, `Income_LogReg_1Page_Summary_Report.docx`, `income_logreg_flowchart.png`.

Work top to bottom. Cells marked `# YOUR CODE HERE` are for you. Peek at the solution notebook only after you have an answer.

## Inline cheat-sheet (keep this cell visible)

See also **`Income_LogReg_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Strip categoricals | `df[c] = df[c].str.strip()` for object columns |
| Class mix | `df.income.value_counts(normalize=True)` |
| Dummies | `X = pd.get_dummies(df[cols], drop_first=True)` |
| Binary target | `y = np.where(df.income == '<=50K', 0, 1)` |
| Split | `train_test_split(X, y, test_size=0.2, random_state=1)` |
| L1 LR (lesson) | `LogisticRegression(C=0.05, penalty='l1', solver='liblinear')` |
| Confusion | `confusion_matrix(y_test, y_pred)` → [[TN, FP], [FN, TP]] |
| Accuracy | `(TN+TP) / n` or `log_reg.score(x_test, y_test)` |
| Coef table | `pd.DataFrame({'var': cols, 'coef': model.coef_[0]})` |
| ROC / AUC | `roc_curve(y, p); roc_auc_score(y, p)` |
| Scale check | compare ranges: age ~17–90 vs capital-gain 0–99,999 |

**Lesson typo:** the starter `feature_cols` listed `hours-per-week` twice. Deduplicate before `get_dummies` — sklearn ≥1.8 rejects duplicate column names.

## 0. Packages

Run this cell first. `penalty='l1'` with `solver='liblinear'` is the lesson specification. On scikit-learn ≥1.8 you may see a deprecation warning; the fit still runs. The new-style equivalent is noted in the alternate section.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load, name columns, strip whitespace

### Task 1.1

The raw file has **no header**. Assign the 15 official UCI names, strip leading/trailing spaces on every string column, and print `head()` plus `shape`.

In [ ]:
col_names = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income",
]
# YOUR CODE HERE
df = None

print(df.head())
print(df.shape)

## 2. EDA and logistic-regression assumptions

### Task 2.1 — class imbalance

The outcome is `income` (`<=50K` vs `>50K`). Print raw counts **and** the normalized mix. Is a 0.50 decision threshold going to look "accurate" for the wrong reason?

In [ ]:
# YOUR CODE HERE
print(df.income.value_counts())
print(df.income.value_counts(normalize=True))

### Task 2.2 — dummy-encode the lesson feature set

Starter list (includes the duplicate on purpose so you can see it):

```python
feature_cols = ['age','capital-gain', 'capital-loss', 'hours-per-week',
                'sex','race', 'hours-per-week', 'education']
```

1. Deduplicate with `dict.fromkeys` or `pd.unique`.
2. Build `X = pd.get_dummies(..., drop_first=True)` and cast to `float` (sklearn 1.9 is picky about bool dummies).
3. Print `X.shape` and `X.columns`.

In [ ]:
feature_cols = [
    "age", "capital-gain", "capital-loss", "hours-per-week",
    "sex", "race", "hours-per-week", "education",
]
# YOUR CODE HERE
feature_cols_u = None
X = None
print(feature_cols_u)
print(X.shape)
print(list(X.columns))

### Task 2.3 — correlation heatmap

`sns.heatmap(X.corr())`. Which dummy blocks are mildly correlated (education levels, race levels)? Does anything look like a 0.99 clone that we must drop?

In [ ]:
# YOUR CODE HERE
plt.figure(figsize=(11, 9))
# heatmap
plt.title("Feature correlation (dummy-encoded X)")
plt.tight_layout()
plt.show()

### Task 2.4 — do we need to scale?  Then encode `y`

Print min / max / mean for `age`, `capital-gain`, `capital-loss`, `hours-per-week`.

- Liblinear L1 **will converge** unscaled, but the continuous coefficients will look tiny next to dummy coefficients.
- Scaling is therefore *interpretability* + *regularization fairness*, not a solver requirement.

Then create `y`: 0 when `income == '<=50K'`, 1 otherwise.

In [ ]:
# YOUR CODE HERE
for c in ["age", "capital-gain", "capital-loss", "hours-per-week"]:
    print(c, X[c].min(), X[c].max(), round(X[c].mean(), 2))

y = None
print("positivity rate", y.mean())

## 3. Fit the lesson model

### Task 3.1 — split + L1 logistic regression

- `random_state=1`, `test_size=0.2`
- `LogisticRegression(C=0.05, penalty='l1', solver='liblinear')`
- `y_pred = log_reg.predict(x_test)`

In [ ]:
# YOUR CODE HERE
x_train = x_test = y_train = y_test = None
log_reg = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
y_pred = None
print(x_train.shape, x_test.shape, y_train.mean(), y_test.mean())

### Task 3.2 — intercept and coefficients

Print `log_reg.intercept_` and `log_reg.coef_`.

In [ ]:
print("Model Parameters, Intercept:")
# YOUR CODE HERE
print("Model Parameters, Coeff:")
# YOUR CODE HERE

### Task 3.3 — confusion matrix and accuracy

Expected ballpark on this split: accuracy near **0.83**, with many more FN than FP on the `>50K` class (the model is conservative at t = 0.5).

In [ ]:
print("Confusion Matrix on test set:")
# YOUR CODE HERE
print("Accuracy Score on test set:")
# YOUR CODE HERE

## 4. Coefficient table and bar plot

### Task 4.1

Build a DataFrame `coef_df` with columns `var`, `coef`. Drop rows with `coef == 0` (L1 sparsity). Sort ascending and print.

In [ ]:
# YOUR CODE HERE
coef_df = None
print(coef_df)

### Task 4.2 — bar plot of the surviving coefficients

Rotate x-tick labels 90°. Title: `LR Coefficient Values`.

In [ ]:
# YOUR CODE HERE
plt.figure(figsize=(10, 7))
plt.xticks(rotation=90)
plt.title("LR Coefficient Values")
plt.tight_layout()
plt.show()

## 5. ROC curve and AUC

`predict_proba` → column 1 → `roc_auc_score` and `roc_curve`. Draw the chance diagonal.

In [ ]:
# YOUR CODE HERE
y_pred_prob = None
roc_auc = None
print("ROC AUC score:", roc_auc)

fpr = tpr = thresholds = None
plt.figure()
# plot ROC + diagonal
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Alternate code (same scientific result)

### Task 6.1 — drop the duplicate *before* get_dummies with `pd.Index.unique`

Rebuild `X_alt` from the original (messy) list without `dict.fromkeys`.

In [ ]:
# YOUR CODE HERE
X_alt = None
print(X_alt.shape, list(X_alt.columns)[:8])

### Task 6.2 — scaled L1 pipeline

`Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=0.05, penalty="l1", solver="liblinear"))])`.

Compare test accuracy and AUC to the unscaled lesson model. Coefficients will change units (now "per 1 SD").

In [ ]:
# YOUR CODE HERE
pipe = None
print("scaled acc / auc")

### Task 6.3 — sklearn L2 default vs lesson L1

Fit `LogisticRegression(max_iter=2000)` (L2, lbfgs). How many coefficients are *exactly* zero compared with L1?

In [ ]:
# YOUR CODE HERE
log_l2 = None
print("L2 n_zero", None)
print("L2 acc", None)

### Task 6.4 — `education-num` instead of education dummies

Replace the education dummy block with the single ordinal `education-num` (1–16). Refit the same L1 spec. Does AUC move?

In [ ]:
# YOUR CODE HERE
feature_cols_ord = ["age", "capital-gain", "capital-loss", "hours-per-week", "sex", "race", "education-num"]
X_ord = None
print("ordinal AUC")

### Task 6.5 — NumPy threshold helper

`predict_at(p, t) = (p >= t).astype(int)`. Sweep t = 0.25, 0.35, 0.50, 0.65 and print precision, recall, FN, FP.

In [ ]:
def predict_at(proba, t=0.5):
    # YOUR CODE HERE
    return None

# YOUR CODE HERE
for t in (0.25, 0.35, 0.50, 0.65):
    pass

## 7. More practice

### Task 7.1 — add `marital-status`

Rebuild X with the lesson columns **plus** `marital-status`. Refit L1. Which new dummy has the largest |coef|? What happens to test AUC?

In [ ]:
# YOUR CODE HERE


### Task 7.2 — `class_weight='balanced'`

Keep the lesson feature set and split. Add `class_weight='balanced'` to the L1 estimator. Report recall of `>50K` vs the default model. Accuracy will usually drop — that is expected.

In [ ]:
# YOUR CODE HERE
log_bal = None
print("balanced recall / acc / auc")

### Task 7.3 — fairness slice

On the *test* fold of the lesson model, compute recall separately for `sex_Male == 1` and `sex_Male == 0`. A gap does not prove discrimination by itself, but it is a required diagnostic when a protected attribute is a feature.

In [ ]:
# YOUR CODE HERE


### Task 7.4 — which metric when?

One sentence each:

* a bank screening applicants for a *premium* product (cost of a false >50K label is high)
* a public-program outreach list that should not miss true high earners
* a dashboard KPI shown to executives who only want "how often are we right" 

In [ ]:
premium_screen = """..."""
outreach = """..."""
exec_kpi = """..."""
print(premium_screen)
print(outreach)
print(exec_kpi)

## 8. Simulation (edit the boxed parameters)

Change `C`, `N`, `NOISE`, `T`, `N_REPS` and re-run. Each replicate:

1. Draws `N` rows with replacement from the full Adult table.
2. Flips a `NOISE` fraction of training labels.
3. Fits the lesson L1 model on an 80/20 split.
4. Records accuracy, recall, AUC at threshold `T`.

Interpret: small `C` → more zeros; label noise → AUC falls; `T` trades recall for precision.

In [ ]:
# --- editable parameters ---
C = 0.05          # inverse L1 strength (smaller = sparser)
N = 8000          # subsample size (max 32561)
N_REPS = 12
NOISE = 0.00      # training-label flip probability
T = 0.50          # decision threshold
SEED = 1
# ---------------------------

rng = np.random.default_rng(SEED)
rows = []
# YOUR CODE HERE — Monte-Carlo loop writing into `rows`
sim = pd.DataFrame(rows)
print(sim.describe().round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, col in zip(axes, ["acc", "recall", "auc"]):
    if col in sim:
        ax.hist(sim[col], bins=8, color="#4c78a8", edgecolor="white")
        ax.set_title(col)
plt.suptitle(f"Income_LogReg simulation  C={C}  N={N}  noise={NOISE}  T={T}")
plt.tight_layout()
plt.show()

## 9. Audience rewrite (from the attached PDFs)

Using Jočys (data literacy, subject knowledge) and McMurrey (expert / technician / executive / nonspecialist), rewrite **one finding**: *unscaled L1 logistic regression on age, capital flows, hours, sex, race and education reaches test accuracy ≈ 0.83 and AUC ≈ 0.85, but recall of >$50K at t = 0.5 is only ≈ 0.41.*

In [ ]:
expert = """..."""
technician = """..."""
executive = """..."""
nonspecialist = """..."""
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)

## 10. When this project is a good fit — and when it is not

Fill ten good-fit settings and five anti-applications. Think about binary outcomes, independent rows, need for coefficients, 1994-era features, protected attributes, and class imbalance.

In [ ]:
good_fit = [
    # "1. ...",
]
not_a_fit = [
    # "1. ...",
]
for row in good_fit:
    print(row)
print("--- not a fit ---")
for row in not_a_fit:
    print(row)

## 11. Done checklist

- [ ] Whitespace stripped; 32,561 × 15
- [ ] Imbalance printed (~0.76 / 0.24)
- [ ] Duplicate `hours-per-week` removed; X is 24 columns
- [ ] Heatmap + scale diagnosis
- [ ] y encoded 0/1
- [ ] L1 model C=0.05 liblinear, intercept + coef printed
- [ ] Confusion matrix + accuracy
- [ ] Non-zero coef table + bar plot
- [ ] ROC + AUC
- [ ] At least two alternates and the simulation cell
- [ ] Audience paragraph + good-fit / not-a-fit lists